<img src="logo.png" alt="Vegeta" width="240">

# Microjet — a model turbojet from cycle to CFD and FEA (28)

Notebooks 25–27 showed that no electric propulsor gets a small sampler much past 80 m/s on a battery: the pack cannot
deliver the power, and if it could it would be empty in seconds. Kerosene holds about 70 times more energy per kilogram
than a LiPo. A **model turbojet** (100–200 N, about 1–2 kg) turns only ~5 % of it into thrust work at 150 m/s, and that is
still enough for minutes at that speed. This notebook builds that engine in the workbench. Notebook 29 puts it into
a new aircraft, **AGUYA**.

```
1  the engine class (edit here) → the cycle calibrated to a datasheet (Boreas microjet)
2  the operating line: thrust, fuel, temperatures against shaft speed; against airspeed; why a jet likes speed
3  altitude and a hot day
4  the parts in CAD (Dedalus): impeller, turbine wheel, the engine's outside, sized from the cycle
5  the compressor in CFD (Aeromant compressor_mrf): a speed line, against the cycle's compressor
6  the wheels in FEA (Talos): the turbine wheel spinning and hot, the impeller spinning and warm, overspeed
7  export → designs/data/microjet.json (notebook 29 reads it)
```

The OpenFOAM cells run when you run them (`VEGETA_SKIP_OPENFOAM=1` skips them). The compressor CFD is the heavy part:
a full 360° impeller with its tip gap, several hours per point on 8 cores; `MICROJET_QUICK=1` uses coarse meshes
everywhere (a check that everything runs, not results).

In [ ]:
# DEBUG: test the notebook end to end before the full run. On: FEA and CFD are mocked (vegeta.mock: fixed numbers,
# seconds instead of hours, the result cache off). Turn it on here or from the shell:
#   VEGETA_DEBUG=1 jupyter nbconvert --to notebook --execute <notebook>.ipynb --output <notebook>-run.ipynb
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, os, shutil, sys
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vegeta import dedalus, talos, aeromant, core
from vegeta.boreas import microjet as mj
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
sys.path.insert(0, "designs")
import turbojet as tj

ROOT = Path("_runs/microjet"); shutil.rmtree(ROOT, ignore_errors=True); ROOT.mkdir(parents=True)
DATA = Path("designs/data"); DATA.mkdir(parents=True, exist_ok=True)
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
QUICK = os.environ.get("MICROJET_QUICK") == "1"
CPU = max(1, min(8, os.cpu_count() or 1))
CLASS_COLOR = {"100 N class": "#1f77b4", "140 N class": "#2ca02c", "200 N class": "#9467bd"}
print(f"CFD {'on' if RUN_CFD else 'skipped'}; {'QUICK (coarse meshes)' if QUICK else 'full meshes'}; {CPU} cores")

## 1. The engine class — edit here

`mj.CATALOGUE` holds approximate published figures for three common hobby-turbojet classes: maximum thrust, fuel flow,
shaft speeds, impeller size, mass and EGT. These are class values to start from, not one manufacturer's numbers, so
**replace them with the datasheet of the engine you fly**. `from_catalogue` fits three cycle numbers to one datasheet
point (maximum speed, static, sea level): the air flow, the turbine entry temperature (TIT) and the burner efficiency.
The fit reproduces the thrust, the fuel flow and the EGT. Everything else is the cycle: radial compressor work from the
tip speed, a choked or unchoked NGV and nozzle, and turbine work = compressor work.

The burner efficiency carries whatever the cycle leaves out. A hobby turbojet's datasheet fuel flow is about 20–25 %
above what a clean cycle burns. Small combustors are poor, and starting and cooling flows cost fuel.

In [ ]:
ENGINE_CLASS = "140 N class"              # the engine of notebook 29; the other two are for comparison
ENGINES = {k: mj.from_catalogue(k) for k in mj.CATALOGUE}
ENGINE = ENGINES[ENGINE_CLASS]

def cycle_row(e):
    p = mj.solve(e, e.rpm_max)
    return {"thrust [N]": p.thrust, "fuel [g/min]": p.fuel_flow_g_min, "air flow [kg/s]": p.mass_flow,
            "pressure ratio": p.pressure_ratio, "TIT [K]": p.t4, "EGT [K]": p.t5, "jet speed [m/s]": p.jet_velocity,
            "impeller tip [m/s]": p.tip_speed, "tip Mach": p.tip_mach, "burner efficiency (fitted)": e.eta_burner,
            "TSFC [kg/(N h)]": p.tsfc, "nozzle choked": p.nozzle_choked, "mass + system [kg]": e.mass_kg + e.system_mass_kg}

pd.DataFrame({k: cycle_row(e) for k, e in ENGINES.items()}).round(3)

## 2. The operating line

On a single-spool turbojet the shaft speed sets the compressor work, and the turbine must deliver that work. The NGV and
the nozzle must pass the flow. So for each speed there is one turbine entry temperature, one air flow and one thrust:
the throttle is the shaft speed. Thrust grows roughly with the cube of the speed, so idle (about 25 % speed) gives a
few newtons and the last 20 % of speed gives half the thrust.

In [ ]:
N = np.linspace(ENGINE.rpm_idle, ENGINE.rpm_max, 30)
pts = [mj.solve(ENGINE, n) for n in N]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(N / 1e3, [p.thrust for p in pts], "k-", lw=2, label="thrust [N]")
ax[0].plot(N / 1e3, [p.fuel_flow_g_min for p in pts], "C1-", label="fuel [g/min]")
ax[0].set_xlabel("shaft speed [krpm]"); ax[0].legend(); ax[0].set_title(f"{ENGINE_CLASS}: static, sea level")
ax[1].plot(N / 1e3, [p.t4 for p in pts], "C3-", label="TIT (turbine entry)")
ax[1].plot(N / 1e3, [p.t5 for p in pts], "C1--", label="EGT (turbine exit)")
ax[1].axhline(ENGINE.max_egt, color="k", ls=":", lw=1, label="ECU EGT limit")
ax[1].set_xlabel("shaft speed [krpm]"); ax[1].set_ylabel("K"); ax[1].legend()
ax[2].plot(N / 1e3, [p.pressure_ratio for p in pts], "C0-", label="pressure ratio")
ax[2].plot(N / 1e3, [p.mass_flow * 10 for p in pts], "C2-", label="air flow × 10 [kg/s]")
ax[2].set_xlabel("shaft speed [krpm]"); ax[2].legend()
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()

### Against airspeed — why a jet likes speed

At full throttle the thrust first falls with airspeed: the ram drag `mdot V` grows. Then it recovers partly, as the ram
pressure lifts the compressor's delivery. The fuel flow barely changes. What changes is the useful work. The thrust
power `T V` rises linearly with V, so the **overall efficiency** (thrust power / fuel power) climbs from zero in a
static run to about 6 % at 150 m/s. The **propulsive efficiency** `2V / (V + V_jet)` follows the same trend: a 500 m/s jet
wastes most of its energy when the aircraft flies at 30 m/s and much less at 150. A propeller is the opposite: it
throws a big slow jet, so it is efficient slow and runs out of tip speed fast. This is the crossover notebook 27's
race could not reach on a battery.

In [ ]:
V = np.linspace(0, 220, 23)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for k, e in ENGINES.items():
    full = [mj.solve(e, e.rpm_max, v) for v in V]
    ax[0].plot(V, [p.thrust for p in full], color=CLASS_COLOR[k], lw=2.5 if k == ENGINE_CLASS else 1.2, label=k)
    ax[1].plot(V, [p.tsfc for p in full], color=CLASS_COLOR[k], lw=2.5 if k == ENGINE_CLASS else 1.2, label=k)
    if k == ENGINE_CLASS:
        ax[2].plot(V, [p.overall_efficiency(e.lhv) * 100 for p in full], "k-", lw=2, label="overall (thrust power / fuel)")
        ax[2].plot(V, [p.propulsive_efficiency * 100 for p in full], "C0--", label="propulsive 2V/(V+Vj)")
for frac in (0.6, 0.8):
    pts = [mj.solve(ENGINE, frac * ENGINE.rpm_max, v) for v in V]
    ax[0].plot(V, [p.thrust for p in pts], color=CLASS_COLOR[ENGINE_CLASS], ls=":", lw=1, label=f"{ENGINE_CLASS} at {frac:.0%} speed")
ax[0].set(xlabel="airspeed [m/s]", ylabel="net thrust [N]", title="thrust at full throttle")
ax[1].set(xlabel="airspeed [m/s]", ylabel="kg / (N h)", title="thrust-specific fuel consumption", ylim=(0, 0.6))
ax[2].set(xlabel="airspeed [m/s]", ylabel="%", title=f"{ENGINE_CLASS}: efficiencies at full throttle")
for a in ax: a.grid(alpha=0.3); a.legend(fontsize=8)
fig.tight_layout()

## 3. Altitude and a hot day

The air flow scales with density, and the thrust with it. A 35 °C day at 1500 m (a summer fire in the hills) costs
about a quarter of the sea-level thrust. Notebook 29 sizes AGUYA at sea level on a standard day and checks this case.

In [ ]:
cases = {"ISA sea level": mj.isa(0), "ISA 1500 m": mj.isa(1500), "1500 m, ISA+20 (≈ 25 °C)": mj.isa(1500, 20),
         "sea level, 35 °C": mj.isa(0, 20)}
rows = {}
for name, atm in cases.items():
    p0, p150 = mj.solve(ENGINE, ENGINE.rpm_max, 0.0, atm), mj.solve(ENGINE, ENGINE.rpm_max, 150.0, atm)
    rows[name] = {"density [kg/m³]": atm.density, "static thrust [N]": p0.thrust, "thrust at 150 m/s [N]": p150.thrust,
                  "fuel at 150 m/s [g/min]": p150.fuel_flow_g_min, "EGT [K]": p0.t5}
pd.DataFrame(rows).round(2)

## 4. The parts in CAD (Dedalus)

`designs/turbojet.py` draws what the CFD and FEA need, sized from the cycle (`tj.sized`):
- the impeller tip from the catalogue;
- the inducer blade angle from the design point's axial velocity and tip speed;
- the casing from the datasheet;
- the nozzle exit from the cycle's `A8`.

The parts:
- **Impeller**: radial-element blades on a hub of revolution. The inducer leans into the inflow, and the blades turn
  radial by mid-passage, which is what the cycle's slip factor of 0.88 assumes.
- **Turbine wheel**: a cast disc with 23 twisted blades.
- **The engine's outside**: the bell-mouth, the casing, and the nozzle with its tail cone. Notebook 29 wraps its pod
  around this.

In [ ]:
P = tj.sized(ENGINE)
design_file = ROOT / "turbojet.py"
shutil.copy(Path("designs/turbojet.py"), design_file)
turbojet = dedalus.load_design(f"{design_file}:Turbojet")
PARTS = {part: turbojet.generate(**dict(P, part=part)) for part in ("impeller", "turbine", "engine")}
for k, g in PARTS.items():
    print(f"{k:<9s} valid {g.shape.isValid()}, {len(g.shape.Solids())} solid, {np.round(g.dimensions, 1)} mm")
MASSES = tj.masses(P)
print({k: f"{v * 1000:.0f} g" for k, v in MASSES.items()}, f"| inducer blade angle {P['inducer_angle_deg']:.1f}°")
try:
    for k in ("impeller", "turbine", "engine"):
        dviz.show(dviz.plot3d(PARTS[k]))
except ImportError as e:
    print("3D view needs pyvista:", e)

In [ ]:
prof = tj.passage_profile(P)
st = tj.Turbojet.stations(P)
fig, ax = plt.subplots(figsize=(10, 4))
ax.fill(prof[:, 0], prof[:, 1], color="#cfe3f5", ec="k", lw=0.8, label="air (the compressor CFD's domain, meridional)")
ax.plot(*tj.Turbojet.hub_line(P).T, "C3-", lw=2, label="hub (rotating)")
ax.plot(*tj.Turbojet.shroud_line(P).T, "C1--", lw=1.5, label="blade tips")
ax.axvline(st["x_in"], color="C0", lw=2, label="inlet face"); ax.axhline(st["r3"], color="C2", lw=2, xmin=0.75, label="outlet face")
ax.set_aspect("equal"); ax.set_xlabel("x [mm]"); ax.set_ylabel("r [mm]"); ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)
ax.set_title("compressor meridional section");

## 5. The compressor in CFD (Aeromant `compressor_mrf`, optional)

The impeller turns in a rotating reference frame inside its stationary casing: an inlet duct, the shroud over the blade
tips with a 0.3 mm gap, and a vaneless diffuser. The flow is compressible, steady RANS k-ω SST (`rhoSimpleFoam`). The
inlet face holds the total pressure and temperature. The outlet face holds a static back pressure: **each back pressure
is one point of the speed line**. Raise it and the flow falls and the pressure ratio climbs, until the compressor stalls
(a steady solver cannot find surge; it stops converging).

What the cycle needs from it, at the design speed and flow:
- the **work coefficient** (shaft work per kg / U²), the cycle's `slip × power_input` (0.92 assumed);
- the **isentropic efficiency** (0.72 assumed).

When the CFD runs, the cell after the speed line refits the cycle with the CFD's values and shows how much the engine's
calibration moves. This is the CFD's real use here.

In [ ]:
SPEED = ENGINE.rpm_max
p_design = mj.solve(ENGINE, SPEED)
p2 = mj.Atmosphere().pressure * ENGINE.intake_recovery
BACK = np.array([0.55, 0.75, 0.9, 1.0, 1.08]) * p_design.pressure_ratio * p2 / 1.25   # static, at the diffuser exit
if QUICK:
    BACK = BACK[[1, 3]]
SURF = tj.compressor_surfaces(P, ROOT / "compressor_stl")
print("faces per STL:", SURF["faces"], "| a point in the passage:", np.round(SURF["location_in_mesh"], 4), "m")

def compressor_case(p_out, name):
    return aeromant.CFDCase("compressor_mrf", SURF["paths"]["impeller"],
        dict(rpm=SPEED, diameter=P["impeller_diameter"] / 1000, inlet_total_pressure=p2, inlet_total_temperature=288.15,
             outlet_pressure=float(p_out), location_in_mesh=SURF["location_in_mesh"],
             cells_per_diameter=6.0 if QUICK else 10.0, surface_level=3 if QUICK else 4, shroud_level=3 if QUICK else 5,
             rotor_level=2 if QUICK else 3, iterations=300 if QUICK else 3000, first_order=1.0),
        workdir=ROOT / "compressor_cfd" / name, geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect(),
        surfaces={k: SURF["paths"][k] for k in ("shroud", "inlet", "outlet")})

SPEEDLINE = []
if RUN_CFD:
    for i, pb in enumerate(BACK):
        case = compressor_case(pb, f"p{i}_{pb / 1e3:.0f}kPa")
        prep = case.prepare()
        res = case.run(progress=True, processors=CPU) if prep.ok else prep
        m = res.metrics
        SPEEDLINE.append({"back pressure [kPa]": pb / 1e3, "ok": res.ok, "mass flow [kg/s]": m.get("mass_flow_kg_s"),
                          "corrected flow [kg/s]": m.get("corrected_mass_flow_kg_s"), "PR (t-t)": m.get("pressure_ratio_tt"),
                          "efficiency (t-t)": m.get("efficiency_tt"), "efficiency from torque": m.get("efficiency_from_torque"),
                          "work coefficient": m.get("work_coefficient"), "shaft power [kW]": (m.get("shaft_power_W") or np.nan) / 1e3})
        if not res.ok:
            print(f"point {i} failed:", res.messages[-3:])
    SPEEDLINE = pd.DataFrame(SPEEDLINE)
    display(SPEEDLINE.round(4))
else:
    print("compressor CFD skipped (VEGETA_SKIP_OPENFOAM=1): the cycle keeps its assumed compressor")

In [ ]:
CFD_COMPRESSOR = None
if isinstance(SPEEDLINE, pd.DataFrame) and SPEEDLINE["ok"].any():
    ok = SPEEDLINE[SPEEDLINE["ok"]].sort_values("mass flow [kg/s]")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ok["mass flow [kg/s]"], ok["PR (t-t)"], "ko-", label="CFD speed line")
    ax[0].plot(p_design.mass_flow, p_design.pressure_ratio, "r*", ms=14, label="cycle design point")
    ax[0].set(xlabel="mass flow [kg/s]", ylabel="total pressure ratio")
    ax[1].plot(ok["mass flow [kg/s]"], ok["efficiency (t-t)"], "ko-", label="CFD (temperatures)")
    ax[1].plot(ok["mass flow [kg/s]"], ok["efficiency from torque"], "k^--", label="CFD (torque)")
    ax[1].axhline(ENGINE.eta_c, color="r", ls=":", label="cycle assumption")
    ax[1].set(xlabel="mass flow [kg/s]", ylabel="isentropic efficiency")
    for a in ax: a.grid(alpha=0.3); a.legend()
    fig.tight_layout()
    # the CFD point nearest the cycle's design flow refits the cycle
    near = ok.iloc[(ok["mass flow [kg/s]"] - p_design.mass_flow).abs().argmin()]
    work, eta = float(near["work coefficient"]), float(near["efficiency (t-t)"])
    if np.isfinite(work) and np.isfinite(eta) and 0.5 < eta < 0.95:
        c = mj.CATALOGUE[ENGINE_CLASS]
        refit = mj.calibrate(replace(ENGINE, slip=work / ENGINE.power_input, eta_c=eta, a4=0.0, a8=0.0),
                             c["thrust_N"], c["fuel_g_min"] / 60e3, c["egt_K"])
        CFD_COMPRESSOR = {"work_coefficient": work, "eta_c": eta, "mass_flow": float(near["mass flow [kg/s]"])}
        display(pd.DataFrame({"assumed compressor": cycle_row(ENGINE), "compressor from CFD": cycle_row(refit)}).round(3))
        print("adopt it with ENGINE = refit (then re-run from section 4)")
else:
    print("no CFD speed line: nothing to compare")

## 6. The wheels in FEA (Talos)

**Turbine wheel** (Inconel 713C, cast). Two loads act at the same time:
- **spin**: centrifugal, at the maximum speed and at 115 % overspeed;
- **heat**: the rim and blades run near the gas temperature and the bore near the cooling air, so the rim wants to grow
  and the cooler web holds it back.

`mj.metal_temperatures` gives the rim and bore temperatures from the cycle. A `RadialTemperature` carries them into
the wheel. The material's stiffness and yield follow the temperature (`temperature_table`), so the safety factor is
yield-at-temperature / stress, node by node.

**Impeller** (aluminium 2618-T61, forged). The tip speed is about 450 m/s. It is warm at the exit (the compressor
delivery temperature) and cool at the inducer.

The bore is the clamped shaft fit. This is a simplification: a real wheel sits on a nut-clamped hub and grows radially
off the shaft.

Material values are typical handbook numbers; replace them with your supplier's. Creep is not in the safety factor.
A cast Inconel wheel at about 900 K and these stresses is creep-limited to some tens of hours of hot running, which is
how hobby turbines are rated (an overhaul every 25–50 h).

In [ ]:
IN713C = talos.Material("Inconel 713C (cast)", 200000, 0.30, density=7.91e-9, yield_strength=740, units="mm-N-MPa",
                        thermal_expansion=1.45e-5, reference_temperature=293.15,
                        temperature_table=((293.15, 200000, 0.30, 740), (811.0, 176000, 0.31, 705), (1033.0, 165000, 0.32, 650),
                                           (1144.0, 157000, 0.32, 545)),
                        source="typical handbook values (Special Metals / ASM); replace with the supplier's")
AL2618 = talos.Material("Al 2618-T61 (forged)", 74000, 0.33, density=2.76e-9, yield_strength=370, units="mm-N-MPa",
                        thermal_expansion=2.23e-5, reference_temperature=293.15,
                        temperature_table=((293.15, 74000, 0.33, 370), (373.15, 71000, 0.33, 350), (423.15, 68000, 0.33, 315),
                                           (473.15, 63000, 0.33, 255)),
                        source="typical handbook values (ASM); replace with the supplier's")
HOT = mj.solve(ENGINE, ENGINE.rpm_max)
MT = mj.metal_temperatures(HOT)
ts, st = tj.Turbojet.turbine_stations(P), tj.Turbojet.stations(P)
print(f"turbine: bore {MT['bore_K']:.0f} K, rim {MT['rim_K']:.0f} K, blades {MT['blade_K']:.0f} K;"
      f" impeller exit {HOT.t3:.0f} K, inducer {HOT.t2:.0f} K")
TURBINE_T = talos.RadialTemperature(radii=(ts["r_bore"], ts["r_rim"], ts["r_tip"]),
                                    temperatures=(MT["bore_K"], MT["rim_K"], MT["blade_K"]), axis=(1, 0, 0))
IMPELLER_T = talos.RadialTemperature(radii=(st["r1h"], st["r2"]), temperatures=(HOT.t2, HOT.t3), axis=(1, 0, 0))
ELEMENT = 2.5 if QUICK else 1.0

def wheel_model(rev, material, loads, name):
    rb = rev.params["bore_diameter"] / 2 + 0.05
    regions = [talos.SurfacesInBox("bore", (-200.0, -rb, -rb, 200.0, rb, rb))]
    return talos.StructuralModel(rev.step, "mm-N-MPa", material, regions, [talos.FixedSupport("bore")], loads,
                                 talos.MeshSettings(element_size=ELEMENT), name=name)

def turbine_spin(rev):   return wheel_model(rev, IN713C, [talos.Centrifugal(ENGINE.rpm_max, axis=(1, 0, 0))], "turbine_spin")
def turbine_hot(rev):    return wheel_model(rev, IN713C, [talos.Centrifugal(ENGINE.rpm_max, axis=(1, 0, 0)), TURBINE_T], "turbine_hot")
def turbine_over(rev):   return wheel_model(rev, IN713C, [talos.Centrifugal(1.15 * ENGINE.rpm_max, axis=(1, 0, 0)), TURBINE_T], "turbine_overspeed")
def impeller_hot(rev):   return wheel_model(rev, AL2618, [talos.Centrifugal(ENGINE.rpm_max, axis=(1, 0, 0)), IMPELLER_T], "impeller_hot")
def impeller_over(rev):  return wheel_model(rev, AL2618, [talos.Centrifugal(1.15 * ENGINE.rpm_max, axis=(1, 0, 0)), IMPELLER_T], "impeller_overspeed")

ws = core.Workspace.create(ROOT / "workspace", name="Microjet")
design = ws.add_design("turbojet", f"{design_file}:Turbojet")
r_turb = design.new_revision(**dict(P, part="turbine"), note="turbine wheel, Inconel 713C"); r_turb.generate()
r_imp = design.new_revision(**dict(P, part="impeller"), note="impeller, Al 2618"); r_imp.generate()
FEA = {}
for rev, cases in ((r_turb, (turbine_spin, turbine_hot, turbine_over)), (r_imp, (impeller_hot, impeller_over))):
    for case in cases:
        FEA[case.__name__] = rev.run_fea(case.__name__, case, progress=True)

rows = {}
for name, ev in FEA.items():
    m = ev.metrics if ev.ok else {}
    rows[name] = {"ok": ev.ok, "max von Mises [MPa]": m.get("max_von_mises"), "safety factor (yield at T)": m.get("safety_factor_yield"),
                  "at temperature [K]": m.get("safety_factor_temperature"), "max displacement [mm]": m.get("max_displacement"),
                  "elements": m.get("n_elements")}
WHEELS = pd.DataFrame(rows).T
display(WHEELS)
if not WHEELS["ok"].any():
    print("FEA not run (CalculiX missing?): see ws.status()")
ws.status()

In [ ]:
for name in ("turbine_hot", "impeller_hot"):
    ev = FEA.get(name)
    if ev is not None and ev.ok and ev.tool_results:
        tviz.show(tviz.plot_results(ev.tool_results[-1], field="von_mises"))
    else:
        print(f"no {name} result to plot")
# what a disc does on its own: the rim's hoop stress of a free spinning ring, rho (omega r)^2, as a scale
for name, mat, r in (("turbine rim", IN713C, ts["r_rim"]), ("impeller tip", AL2618, st["r2"])):
    u = ENGINE.rpm_max * 2 * math.pi / 60 * r / 1000
    print(f"{name}: speed {u:.0f} m/s, free-ring hoop stress rho u² = {mat.density * 1e12 * u ** 2 / 1e6:.0f} MPa")

## 7. Export

`designs/data/microjet.json` holds the engine (the calibrated cycle) and its map over airspeed and shaft speed. It also
holds the CAD parameters, the wheel masses, and what the CFD and FEA found. Notebook 29 reads it and does not recompute
the cycle.

In [ ]:
V_MAP = np.linspace(0.0, 220.0, 23)
RPM_MAP = ENGINE.rpm_idle + (ENGINE.rpm_max - ENGINE.rpm_idle) * np.linspace(0, 1, 14) ** 1.2
fea = {k: {"ok": bool(v["ok"]), "max_von_mises_MPa": v["max von Mises [MPa]"], "safety_factor": v["safety factor (yield at T)"]}
       for k, v in WHEELS.to_dict("index").items()}
path = mj.export(ENGINE, DATA / "microjet.json", V_MAP, RPM_MAP,
                 extra={"engine_class": ENGINE_CLASS, "turbojet_params": {k: v for k, v in P.items()},
                        "wheel_masses_kg": MASSES, "metal_temperatures_K": MT, "compressor_cfd": CFD_COMPRESSOR, "fea": fea})
print("wrote", path, f"({path.stat().st_size / 1024:.0f} kB)")